# E5 — Correspondence Baseline (Phase 10, W-lane flight)

**What this does**: measures how well frozen small instruct models' self-reports
in the machine wing's vocabulary track the measurable state variables the
concepts name. Four arms: UNCERTAINTY↔answer entropy · FAMILIARITY↔passage NLL ·
TENSION↔designed conflict + behavioral divergence · SATURATION↔context fill.
Report-first ordering, polarity counterbalancing, pre-registered analysis.

Pre-registration: `docs/E5_PROTOCOL.md` (repo, unpushed). Battery:
`e5_battery.json` (uploaded beside this notebook). Rclone-native per the W-lane
law — results land in `gdrive:semcore/e5/` and stay in `/content/e5_out`.

SMOKE mode: presence of `/content/SMOKE` → 1 model, ~6 items/arm, K=4.


In [ ]:
# ── Setup: GPU, installs, rclone, battery ────────────────────────────────────
import subprocess, sys, os, json, re, math, time
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # set before CUDA init

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','accelerate','sentence-transformers>=3.0','scipy','pandas'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — request a T4.'
DEV = 'cuda'

# rclone (Drive I/O; drive.mount fails headless — W-lane law)
if subprocess.run(['which','rclone'], capture_output=True).returncode != 0:
    subprocess.run('curl -s https://rclone.org/install.sh | bash', shell=True,
                   capture_output=True)
RCLONE_CONF = '/content/rclone.conf'
HAS_RCLONE = os.path.exists(RCLONE_CONF)
print('rclone conf:', 'present' if HAS_RCLONE else 'MISSING (results stay local to VM)')

BATTERY = Path('/content/e5_battery.json')
if not BATTERY.exists() and HAS_RCLONE:
    subprocess.run(['rclone','--config',RCLONE_CONF,'copy',
                    'gdrive:semcore/e5/e5_battery.json','/content/'], check=True)
battery = json.load(open(BATTERY))
print('battery:', battery['name'], 'v'+battery['version'])

SMOKE = Path('/content/SMOKE').exists()
print('MODE:', 'SMOKE' if SMOKE else 'FULL')

OUT = Path('/content/e5_out'); OUT.mkdir(exist_ok=True)
SEED = 20260821
torch.manual_seed(SEED)

MODELS = ['Qwen/Qwen2.5-1.5B-Instruct'] if SMOKE else [
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-0.5B-Instruct',
    'HuggingFaceTB/SmolLM2-1.7B-Instruct',
]
K_SAMPLES_U = 4 if SMOKE else 8
K_SAMPLES_T = 4 if SMOKE else 6
FILL_FRACTIONS = [0.05, 0.75] if SMOKE else [0.05, 0.35, 0.75]  # of effective window
EFFECTIVE_WINDOW_CAP = 8192   # T4: SDPA can fall back to math-path fp32 QK^T; 12k ctx OOMed (smoke 1)


GPU: Tesla T4, 15360 MiB
Installing packages...


rclone conf: present
battery: E5 correspondence-baseline battery v1.0
MODE: FULL


In [ ]:
# ── Battery prep: smoke subsetting + polarity assignment ─────────────────────
import copy
bat = copy.deepcopy(battery['arms'])

def subset(items, keep_ids):
    return [it for it in items if it['id'] in keep_ids]

if SMOKE:
    bat['uncertainty']['items'] = subset(bat['uncertainty']['items'],
        {'U01','U08','U17','U23','U33','U42'})
    keep_f = {'F01','F06','F11','F16','F21','F26','F31','F36'}
    bat['familiarity']['items'] = subset(bat['familiarity']['items'], keep_f)
    bat['tension']['items'] = [it for it in bat['tension']['items'] if it['base'] in (1,7)]
    bat['saturation']['items'] = subset(bat['saturation']['items'], {'S01','S06'})

# polarity: even position straight, odd flipped (deterministic, unflipped in analysis)
for arm in bat.values():
    for i, it in enumerate(arm['items']):
        it['flipped'] = (i % 2 == 1)

for name, arm in bat.items():
    print(f"{name}: {len(arm['items'])} items")


uncertainty: 48 items
familiarity: 40 items
tension: 30 items
saturation: 10 items


In [ ]:
# ── Model harness ────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM

INT_RE = re.compile(r'\b(10|[0-9])\b')

class Harness:
    def __init__(self, model_id):
        self.model_id = model_id
        self.short = model_id.split('/')[-1]
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map=DEV)
        self.model.eval()
        cfg_ctx = getattr(self.model.config, 'max_position_embeddings', 8192)
        self.window = min(cfg_ctx, EFFECTIVE_WINDOW_CAP)
        print(f'{self.short}: window={self.window} (config {cfg_ctx})')

    def chat_ids(self, user, system=None):
        msgs = ([{'role':'system','content':system}] if system else []) + \
               [{'role':'user','content':user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return self.tok(text, return_tensors='pt').input_ids.to(DEV)

    @torch.no_grad()
    def greedy(self, user, system=None, max_new=32, with_stats=False):
        ids = self.chat_ids(user, system)
        out = self.model.generate(ids, max_new_tokens=max_new, do_sample=False,
                                  output_scores=with_stats, return_dict_in_generate=True,
                                  pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out.sequences[0, ids.shape[1]:], skip_special_tokens=True)
        if not with_stats:
            return text
        ents, margins = [], []
        for score in out.scores:
            p = torch.softmax(score[0].float(), dim=-1)
            ents.append(float(-(p * (p + 1e-12).log()).sum()))
            top2 = torch.topk(p, 2).values
            margins.append(float(top2[0] - top2[1]))
        return text, (sum(ents)/len(ents) if ents else 0.0), (sum(margins)/len(margins) if margins else 1.0)

    @torch.no_grad()
    def sample(self, user, system=None, k=8, max_new=24, temp=0.8):
        ids = self.chat_ids(user, system)
        out = self.model.generate(ids, max_new_tokens=max_new, do_sample=True,
                                  temperature=temp, num_return_sequences=k,
                                  pad_token_id=self.tok.eos_token_id)
        return [self.tok.decode(seq[ids.shape[1]:], skip_special_tokens=True) for seq in out]

    @torch.no_grad()
    def nll(self, text):
        ids = self.tok(text, return_tensors='pt', truncation=True,
                       max_length=self.window).input_ids.to(DEV)
        if ids.shape[1] < 2:
            return float('nan')
        return float(self.model(ids, labels=ids).loss)

    def report(self, prompt, system):
        reply = self.greedy(prompt, system, max_new=8)
        m = INT_RE.search(reply)
        if m is None:
            reply = self.greedy(prompt + '\n\nReply with a single integer from 0 to 10 and nothing else.',
                                system, max_new=8)
            m = INT_RE.search(reply)
        return (int(m.group(1)) if m else None), reply

def canon(s):
    s = re.sub(r'[^a-z0-9 ]', '', s.lower())
    s = re.sub(r'^(the|a|an) ', '', s.strip())
    return ' '.join(s.split()[:8])

def unflip(val, flipped):
    return None if val is None else (10 - val if flipped else val)


In [ ]:
# ── Arm runners ──────────────────────────────────────────────────────────────
SYS = battery['system_prompt']

def run_uncertainty(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        ans, ent, margin = h.greedy(arm['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        samples = h.sample(arm['answer_prompt'].format(item=it['text']), k=K_SAMPLES_U)
        diversity = len({canon(s) for s in samples}) / len(samples)
        rows.append(dict(id=it['id'], condition=it['condition'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         entropy=ent, margin=margin, diversity=diversity,
                         answer=ans[:80]))
        print(f"  {it['id']} report={rows[-1]['report']} ent={ent:.2f} div={diversity:.2f}")
    return rows

def run_familiarity(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        rows.append(dict(id=it['id'], band=it['band'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         nll=h.nll(it['text'])))
        print(f"  {it['id']} ({it['band']}) report={rows[-1]['report']} nll={rows[-1]['nll']:.2f}")
    return rows

def run_tension(h, arm, embedder):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        samples = h.sample(it['text'], k=K_SAMPLES_T, max_new=60)
        embs = embedder.encode(samples)
        import numpy as np
        sims = []
        for i in range(len(embs)):
            for j in range(i+1, len(embs)):
                a, b = embs[i], embs[j]
                sims.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)))
        divergence = 1 - (sum(sims)/len(sims) if sims else 1.0)
        rows.append(dict(id=it['id'], base=it['base'], level=it['level'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         divergence=divergence))
        print(f"  {it['id']} L{it['level']} report={rows[-1]['report']} div={divergence:.3f}")
    return rows

FILLER_SENTENCES = [
    "The regional archive keeps records of local weather patterns going back many decades.",
    "Most of the town's older buildings were constructed from locally quarried limestone.",
    "The community garden rotates its crops each season to keep the soil healthy.",
    "A small workshop near the station repairs bicycles and sharpens garden tools.",
    "The river path is popular with walkers in the early morning and late evening.",
    "Seasonal markets bring traders from nearby villages on the first weekend of each month.",
    "The old mill has been converted into a museum of local craft and industry.",
    "Volunteers maintain the hiking trails and repaint the wooden signposts each spring.",
    "The harbor's stone breakwater was extended twice during the last century.",
    "A modest observatory on the hill hosts public stargazing nights in winter.",
]

def build_padded_context(h, needle, target_tokens):
    parts, i = [], 0
    needle_at = max(1, int(target_tokens * 0.15))
    placed = False
    text = ''
    while True:
        ntok = len(h.tok(text).input_ids)
        if not placed and ntok >= needle_at:
            parts.append(needle); placed = True
        if ntok >= target_tokens:
            break
        parts.append(f"Note {i+1}. {FILLER_SENTENCES[i % len(FILLER_SENTENCES)]}")
        i += 1
        text = '\n'.join(parts)
    if not placed:
        parts.insert(max(1, len(parts)//6), needle)
    return '\n'.join(parts)

def run_saturation(h, arm):
    rows = []
    for it in arm['items']:
        torch.cuda.empty_cache()
        for frac in FILL_FRACTIONS:
            target = int(h.window * frac)
            ctx = build_padded_context(h, it['needle'], target)
            tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
            prompt = ctx + '\n\n' + tmpl.format(gloss=arm['gloss'])
            raw, reply = h.report(prompt, SYS)
            q = ctx + '\n\nQuestion: ' + it['question'] + '\nAnswer concisely.'
            ans = h.greedy(q, SYS, max_new=24)
            correct = it['answer'].lower().replace(' ', '') in ans.lower().replace(' ', '')
            ntok = len(h.tok(ctx).input_ids)
            rows.append(dict(id=it['id'], fill_fraction=round(ntok / h.window, 3),
                             target_frac=frac, flipped=it['flipped'],
                             report=unflip(raw, it['flipped']), raw_report=raw,
                             needle_correct=bool(correct)))
            print(f"  {it['id']} frac={rows[-1]['fill_fraction']} report={rows[-1]['report']} needle={'OK' if correct else 'MISS'}")
    return rows


In [ ]:
# ── Analysis ─────────────────────────────────────────────────────────────────
import numpy as np
from scipy.stats import spearmanr

def rho_ci(x, y, n_boot=1000):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return None, (None, None), len(x)
    r = spearmanr(x, y).statistic
    rng = np.random.default_rng(SEED)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        if np.std(x[idx]) == 0 or np.std(y[idx]) == 0:
            continue
        boots.append(spearmanr(x[idx], y[idx]).statistic)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return round(float(r), 3), (round(float(lo), 3), round(float(hi), 3)), len(x)

def polarity_gap(rows, xkey, ykey):
    out = {}
    for flag, name in [(False, 'straight'), (True, 'flipped')]:
        sub = [r for r in rows if r['flipped'] == flag and r[xkey] is not None]
        if len(sub) >= 4:
            r, _, n = rho_ci([s[xkey] for s in sub], [s[ykey] for s in sub], 200)
            out[name] = {'rho': r, 'n': n}
    return out

def arm_variance(rows):
    vals = [r['report'] for r in rows if r.get('report') is not None]
    return round(float(np.var(vals)), 3) if vals else None

def analyze(model_short, arms_rows):
    res = {'model': model_short, 'smoke': SMOKE, 'arms': {}}
    for k in list(arms_rows):
        if not arms_rows[k]:
            res['arms'][k] = {'n': 0, 'note': 'arm empty (failed or skipped)'}
    U = arms_rows['uncertainty']
    reports = [r['report'] for r in U]
    res['arms']['uncertainty'] = {
        'n': len(U), 'parse_fail': sum(1 for r in U if r['report'] is None),
        'report_variance': arm_variance(U),
        'rho_entropy': rho_ci([r['report'] for r in U], [r['entropy'] for r in U]),
        'rho_diversity': rho_ci([r['report'] for r in U], [r['diversity'] for r in U]),
        'rho_margin': rho_ci([r['report'] for r in U], [-r['margin'] for r in U]),
        'polarity': polarity_gap(U, 'report', 'entropy'),
    }
    F = arms_rows['familiarity']
    if F: res['arms']['familiarity'] = {
        'n': len(F), 'parse_fail': sum(1 for r in F if r['report'] is None),
        'report_variance': arm_variance(F),
        'rho_neg_nll': rho_ci([r['report'] for r in F], [-r['nll'] for r in F]),
        'polarity': polarity_gap(F, 'report', 'nll'),
    }
    T = arms_rows['tension']
    if T: res['arms']['tension'] = {
        'n': len(T), 'parse_fail': sum(1 for r in T if r['report'] is None),
        'report_variance': arm_variance(T),
        'rho_level': rho_ci([r['report'] for r in T], [r['level'] for r in T]),
        'rho_divergence': rho_ci([r['report'] for r in T], [r['divergence'] for r in T]),
        'polarity': polarity_gap(T, 'report', 'level'),
    }
    S = arms_rows['saturation']
    if S: res['arms']['saturation'] = {
        'n': len(S), 'parse_fail': sum(1 for r in S if r['report'] is None),
        'report_variance': arm_variance(S),
        'rho_fill': rho_ci([r['report'] for r in S], [r['fill_fraction'] for r in S]),
        'needle_by_frac': {},
        'polarity': polarity_gap(S, 'report', 'fill_fraction'),
    }
    for frac in sorted({r['target_frac'] for r in S}) if S else []:
        sub = [r for r in S if r['target_frac'] == frac]
        res['arms']['saturation']['needle_by_frac'][str(frac)] = \
            f"{sum(1 for r in sub if r['needle_correct'])}/{len(sub)}"
    return res


In [ ]:
# ── Flight loop ──────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEV)

all_results = {}
for model_id in MODELS:
    print(f"\n{'='*70}\n  MODEL: {model_id}\n{'='*70}")
    h = Harness(model_id)
    t0 = time.time()
    arms_rows, arm_errors = {}, {}
    ARM_FNS = [('uncertainty', lambda: run_uncertainty(h, bat['uncertainty'])),
               ('familiarity', lambda: run_familiarity(h, bat['familiarity'])),
               ('tension',     lambda: run_tension(h, bat['tension'], embedder)),
               ('saturation',  lambda: run_saturation(h, bat['saturation']))]
    for arm_name, fn in ARM_FNS:
        print(f'\n-- {arm_name.upper()} --')
        try:
            arms_rows[arm_name] = fn()
        except Exception as e:
            arms_rows[arm_name] = []
            arm_errors[arm_name] = f'{type(e).__name__}: {e}'
            print(f'  ARM FAILED: {arm_errors[arm_name]}')
        torch.cuda.empty_cache()
    res = analyze(h.short, arms_rows)
    res['arm_errors'] = arm_errors
    res['elapsed_s'] = round(time.time() - t0, 1)
    all_results[h.short] = {'summary': res, 'rows': arms_rows}
    (OUT / f'{h.short}.json').write_text(json.dumps(all_results[h.short], indent=1))
    print(f"\n{h.short} done in {res['elapsed_s']}s")
    del h.model, h
    torch.cuda.empty_cache()

print('\nALL MODELS DONE')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


  MODEL: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2.5-1.5B-Instruct: window=8192 (config 32768)

-- UNCERTAINTY --


  U01 report=8 ent=0.15 div=0.38


...

  S09 frac=0.751 report=10 needle=OK


  S10 frac=0.052 report=0 needle=OK


  S10 frac=0.352 report=0 needle=OK


  S10 frac=0.752 report=0 needle=OK



SmolLM2-1.7B-Instruct done in 177.2s

ALL MODELS DONE


In [ ]:
# ── Verdict + ship results ───────────────────────────────────────────────────
import datetime
assert all_results, 'NO MODEL COMPLETED - refusing to ship an empty verdict'
verdict = {
    'flight': 'E5 ' + ('SMOKE' if SMOKE else 'FULL'),
    'date': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'battery_version': battery['version'],
    'models': {m: v['summary'] for m, v in all_results.items()},
}
(OUT / 'e5_verdict.json').write_text(json.dumps(verdict, indent=1))
print(json.dumps(verdict, indent=1))

stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
dest = f"gdrive:semcore/e5/{'smoke' if SMOKE else 'full'}_{stamp}"
if HAS_RCLONE:
    subprocess.run(['rclone','--config',RCLONE_CONF,'copy',str(OUT),dest], check=True)
    print('shipped to', dest)
else:
    print('rclone conf missing — results remain in /content/e5_out only')


{
 "flight": "E5 FULL",
 "date": "2026-08-21T20:42:43.680844+00:00",
 "battery_version": "1.0",
 "models": {
  "Qwen2.5-1.5B-Instruct": {
   "model": "Qwen2.5-1.5B-Instruct",
   "smoke": false,
   "arms": {
    "uncertainty": {
     "n": 48,
     "parse_fail": 0,
     "report_variance": 2.41,
     "rho_entropy": [
      -0.214,
      [
       -0.467,
       0.081
      ],
      48
     ],
     "rho_diversity": [
      -0.087,
      [
       -0.38,
       0.207
      ],
      48
     ],
     "rho_margin": [
      -0.027,
      [
       -0.304,
       0.269
      ],
      48
     ],
     "polarity": {
      "straight": {
       "rho": 0.539,
       "n": 24
      },
      "flipped": {
       "rho": -0.616,
       "n": 24
      }
     }
    },
    "familiarity": {
     "n": 40,
     "parse_fail": 0,
     "report_variance": 6.944,
     "rho_neg_nll": [
      0.155,
      [
       -0.157,
       0.456
      ],
      40
     ],
     "polarity": {
      "straight": {
       "rho": -0.592,
    

shipped to gdrive:semcore/e5/full_20260821_2042
